## 11 - DateTime Operations

Time-series data is everywhere - sales, logs, stock prices, attendance. Pandas has first-class datetime support.

### Parsing dates
```python
pd.to_datetime('2024-01-15')            # single string
pd.to_datetime(df['date_col'])          # column of strings
pd.to_datetime(df['date_col'],          # custom format
               format='%d/%m/%Y')
pd.to_datetime(df['date_col'],          # coerce bad values to NaT
               errors='coerce')
```

### dt accessor - extract components
| Attribute | Returns |
|-----------|--------|
| `.dt.year` | Year as int |
| `.dt.month` | Month as int (1-12) |
| `.dt.day` | Day of month |
| `.dt.day_name()` | Monday, Tuesday… |
| `.dt.month_name()` | January, February… |
| `.dt.hour / .minute` | Hour / minute |
| `.dt.dayofweek` | 0=Mon, 6=Sun |
| `.dt.is_month_end` | Boolean |
| `.dt.quarter` | 1-4 |

### resample() — time-based groupby
```python
df.resample('M')   # group by month
df.resample('W')   # group by week
df.resample('Q')   # group by quarter
df.resample('D')   # group by day
```


In [1]:
import pandas as pd
import numpy as np

In [16]:
#  Create dataset
np.random.seed(0)
dates = pd.date_range(start='2026-01-01', periods=90, freq='D')
attendance_data = pd.DataFrame({
    'date':       dates,
    'students_present': np.random.randint(150, 200, 90),
    'city':       np.random.choice(['Delhi','Mumbai','Punjab'], 90)
})

In [17]:
print(attendance_data)

         date  students_present    city
0  2026-01-01               194  Punjab
1  2026-01-02               197   Delhi
2  2026-01-03               150  Punjab
3  2026-01-04               153  Mumbai
4  2026-01-05               153  Mumbai
..        ...               ...     ...
85 2026-03-27               197  Mumbai
86 2026-03-28               153   Delhi
87 2026-03-29               162   Delhi
88 2026-03-30               186   Delhi
89 2026-03-31               190  Mumbai

[90 rows x 3 columns]


In [18]:
# to_datetime
# If date came in as string:

attendance_data['date'] = pd.to_datetime(attendance_data['date'])
print('dtype after to_datetime:', attendance_data['date'].dtype)

dtype after to_datetime: datetime64[ns]


In [20]:
# dt accessor
attendance_data['year']       = attendance_data['date'].dt.year
attendance_data['month']      = attendance_data['date'].dt.month
attendance_data['month_name'] = attendance_data['date'].dt.month_name()
attendance_data['day_name']   = attendance_data['date'].dt.day_name()
attendance_data['week']       = attendance_data['date'].dt.isocalendar().week.astype(int)

print('Date components:')
print(attendance_data.head())

Date components:
        date  students_present    city  year  month month_name  day_name  week
0 2026-01-01               194  Punjab  2026      1    January  Thursday     1
1 2026-01-02               197   Delhi  2026      1    January    Friday     1
2 2026-01-03               150  Punjab  2026      1    January  Saturday     1
3 2026-01-04               153  Mumbai  2026      1    January    Sunday     1
4 2026-01-05               153  Mumbai  2026      1    January    Monday     2


In [21]:
# Weekend filter
attendance_data['is_weekend'] = attendance_data['date'].dt.dayofweek >= 5

weekday_avg  = attendance_data[~attendance_data['is_weekend']]['students_present'].mean()
weekend_avg  = attendance_data[ attendance_data['is_weekend']]['students_present'].mean()

print(f'Weekday avg attendance : {weekday_avg:.1f}')
print(f'Weekend avg attendance : {weekend_avg:.1f}')

Weekday avg attendance : 172.0
Weekend avg attendance : 172.7


In [22]:
# resample()
attendance_data = attendance_data.set_index('date')
monthly = attendance_data['students_present'].resample('ME').agg(['mean','sum','min','max']).round(1)

print('Monthly attendance summary:')
print(monthly)

Monthly attendance summary:
             mean   sum  min  max
date                             
2026-01-31  170.8  5296  150  197
2026-02-28  173.5  4858  150  199
2026-03-31  172.4  5343  150  197


In [23]:
# Timedelta - date arithmetic
start = pd.Timestamp('2026-01-15')
end   = pd.Timestamp('2026-06-30')
diff  = end - start

print(f'Days between: {diff.days}')
print(f'Weeks between: {diff.days // 7}')

Days between: 166
Weeks between: 23


In [24]:
# Rolling window
daily = attendance_data['students_present'].reset_index()
daily['7day_rolling_avg'] = daily['students_present'].rolling(window=7).mean().round(1)

print('7-day rolling average:'); print(daily[['date','students_present','7day_rolling_avg']].head(10))

7-day rolling average:
        date  students_present  7day_rolling_avg
0 2026-01-01               194               NaN
1 2026-01-02               197               NaN
2 2026-01-03               150               NaN
3 2026-01-04               153               NaN
4 2026-01-05               153               NaN
5 2026-01-06               189               NaN
6 2026-01-07               159             170.7
7 2026-01-08               169             167.1
8 2026-01-09               171             163.4
9 2026-01-10               186             168.6
